# Verify corrected divacancy energies from raw evidence

For each pristine cell with N atoms and defect with N−2 atoms, the total two-vacancy formation energy is **E(N−2) − ((N−2)/N) E(N)**. This notebook verifies initial removed sites, trajectory and raw-output energies, reference settings and the final **combined atom/cell optimizer force**. An existing result.json or small atomic force alone is insufficient.

The default source is the corrected 20260831 fixed-[110] package. Override with `AL_DEFECTS_DIVACANCY_ROOT` to audit another package. Historical evidence is read only.


In [ ]:
from pathlib import Path
import os
import sys

candidates = ([Path(os.environ['AL_DEFECTS_REPO'])] if os.environ.get('AL_DEFECTS_REPO') else [])
candidates += [Path.cwd(), *Path.cwd().parents]
REPO = next((p.resolve() for p in candidates if (p / 'scripts' / 'divacancy_analysis_checks.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Open this notebook from the repository, or set AL_DEFECTS_REPO to the full repository')
sys.path.insert(0, str(REPO / 'scripts'))
from divacancy_analysis_checks import latest_dftpy_root
import pandas as pd
from collect_dftpy_conventional_vacancy import collect_scan
ROOT = latest_dftpy_root(REPO)
print('Read-only source:', ROOT)


In [ ]:
rows = collect_scan(ROOT, 'pair_scan')
if not rows:
    raise FileNotFoundError(f'No pair cases found in {ROOT}')
table = pd.DataFrame(rows)
columns = ['setting', 'status', 'pair_distance_verified_A', 'pair_direction_verified',
           'Ef_recomputed_eV', 'pristine_combined_fmax_eV_A', 'vacancy_combined_fmax_eV_A',
           'qualification_reasons']
display(table.reindex(columns=columns))
print('Status counts:', table.status.value_counts().to_dict())


`qualified` means the recorded numerical/evidence checks pass. It does not establish thesis acceptance, finite-size convergence, or universal behavior in other crystal directions. The three corrected formation energies round to 1.2784, 1.3322 and 1.3349 eV; the 6–7 Å dip from mixed directions must not be reintroduced.

**Do not infer a formal binding energy from the historical monovacancy 600 eV / 0.250343 Å scan and this divacancy 0.20 Å scan.** Their grids differ. Combining mono/divacancy energies requires matching code, pseudopotential hash, XC, KEDF and weights, grid, cell/reference convention, pressure and convergence tolerance.
